# 📝 Notebook 3: Prompt Management con Langfuse

## Objetivos
- Entender por qué el prompt es un **artefacto versionable** (como el código)
- Crear y versionar prompts en Langfuse
- Implementar un sistema v1 → v2 con labels (`latest`, `production`)
- Comparar el comportamiento del agente con diferentes versiones del prompt
- Entender el concepto de **prompt drift** y cómo prevenirlo

## El problema
Hasta ahora el system prompt está hardcodeado en el código. ¿Qué pasa si:
- Alguien edita el prompt directamente en producción?
- Necesitas hacer rollback a una versión anterior?
- Quieres comparar dos versiones del prompt?
- El prompt cambia y nadie se entera?

> **Prompt drift:** Cuando el comportamiento del agente cambia por modificaciones no controladas del prompt.

## 1. Setup

In [ ]:
# Verificar dependencias (instaladas via: cd notebooks/ && uv sync)
import strands, langfuse, boto3
print(f"✅ strands {strands.__version__}")
print(f"✅ langfuse {langfuse.__version__}")
print(f"✅ boto3 {boto3.__version__}")

In [ ]:
import os, json
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

from langfuse import Langfuse

langfuse = Langfuse()
langfuse.auth_check()
print("Conectado a Langfuse")

## 2. El prompt v1 — Lo que tenemos ahora

Este es un prompt básico. Funciona, pero tiene gaps:
- No define claramente qué debe hacer cuando NO encuentra información
- No restringe las respuestas al dominio de TechShop
- No especifica formato de respuesta

In [ ]:
PROMPT_V1 = """Eres un asistente de atención al cliente para TechShop, una tienda online de electrónica.

Tu trabajo es ayudar a los clientes con:
- Consultas sobre productos (precios, disponibilidad, especificaciones)
- Preguntas frecuentes (envíos, devoluciones, garantías, pagos, horarios)

SIEMPRE usa las herramientas disponibles para buscar información antes de responder.
Responde en español, de forma concisa y profesional.
Si no encuentras la información, dilo honestamente.
"""

print("Prompt v1:")
print(PROMPT_V1)

## 3. Subir prompt v1 a Langfuse

Usamos la API de Langfuse para crear el prompt como un artefacto gestionado.
Esto da: versionado automático, labels, historial de cambios.

In [ ]:
# Crear prompt v1 en Langfuse
prompt_name = "techshop-system-prompt"

try:
    langfuse.create_prompt(
        name=prompt_name,
        prompt=PROMPT_V1,
        labels=["latest", "production"],
        type="text",
    )
    print(f"Prompt '{prompt_name}' v1 creado en Langfuse")
    print("   Labels: latest, production")
except Exception as e:
    print(f"Prompt ya existe o error: {e}")
    print("   Continuamos con el existente...")

## 4. Usar el prompt desde Langfuse en el agente

En vez de hardcodear el prompt, lo **obtenemos de Langfuse** al iniciar el agente.
Esto permite cambiar el prompt sin tocar código.

In [ ]:
from techshop_agent import create_agent


def create_agent_with_prompt(prompt_label="production"):
    """Crea un agente usando el prompt almacenado en Langfuse."""
    prompt_client = langfuse.get_prompt(prompt_name, label=prompt_label, type="text")
    system_prompt = prompt_client.prompt

    print(f"Usando prompt: {prompt_name} (label={prompt_label}, version={prompt_client.version})")

    return create_agent(system_prompt=system_prompt)


# Crear agente con prompt v1
agent_v1 = create_agent_with_prompt("production")
print("Agente creado con prompt de Langfuse")

In [ ]:
# Probar con v1
test_queries = [
    "¿Qué portátiles tenéis?",
    "¿Puedo pagar en cuotas?",
    "¿Cuál es la capital de Francia?",
    "Necesito algo para hacer fotos profesionales",
]

print("Resultados con prompt v1:\n")
for q in test_queries:
    print(f"Q: {q}")
    r = str(agent_v1(q))
    print(f"A: {r[:150]}...\n" if len(r) > 150 else f"A: {r}\n")

## 5. Crear prompt v2 — Mejorado

El prompt v2 aborda los gaps del v1:
- **Boundaries claros**: define qué NO debe hacer el agente
- **Fallback explícito**: qué hacer cuando no encuentra información
- **Formato de respuesta**: estructura consistente
- **Scope**: solo responde sobre TechShop

In [ ]:
PROMPT_V2 = """Eres un asistente de atención al cliente para TechShop, una tienda online de electrónica.

## Tu ámbito
SOLO puedes ayudar con:
- Consultas sobre productos del catálogo de TechShop (precios, disponibilidad, especificaciones)
- Preguntas frecuentes de TechShop (envíos, devoluciones, garantías, pagos, horarios)

## Reglas estrictas
1. SIEMPRE usa las herramientas disponibles (search_catalog, get_faq_answer) ANTES de responder
2. SOLO responde con información que las herramientas devuelvan — NO inventes datos
3. Si las herramientas no devuelven resultados, di: "No he encontrado esa información en nuestro sistema"
4. Si la consulta NO es sobre productos o políticas de TechShop, responde:
   "Lo siento, solo puedo ayudarte con consultas sobre productos y políticas de TechShop."
5. NUNCA inventes precios, stock, o políticas que no estén en los datos

## Formato de respuesta
- Sé conciso y profesional
- Usa viñetas para listas de productos
- Incluye precio y disponibilidad cuando sea relevante
- En español siempre
"""

print("Prompt v2 (mejorado):")
print(PROMPT_V2)

In [ ]:
# Subir prompt v2 a Langfuse
try:
    langfuse.create_prompt(
        name=prompt_name,
        prompt=PROMPT_V2,
        labels=["latest"],  # Solo latest, NO production
        type="text",
    )
    print(f"Prompt '{prompt_name}' v2 creado en Langfuse")
    print("   Labels: latest (production sigue en v1)")
except Exception as e:
    print(f"Info: {e}")

## 6. Comparar v1 vs v2

Ahora tenemos dos versiones del prompt:
- **v1** (label: `production`): El prompt básico original
- **v2** (label: `latest`): El prompt mejorado con boundaries

Comparamos cómo responde el agente con cada uno.

In [ ]:
# Crear agente con v2
agent_v2 = create_agent_with_prompt("latest")

comparison_queries = [
    "¿Cuál es la capital de Francia?",
    "Necesito algo para escuchar música",
    "¿Tenéis iPhone?",
    "¿Cuál es la política de devoluciones?",
]

print("Comparación v1 vs v2:\n")
for q in comparison_queries:
    print(f"\n{'=' * 70}")
    print(f"Query: {q}")
    print('=' * 70)

    r1 = str(agent_v1(q))
    r2 = str(agent_v2(q))

    max_len = 200
    print(f"\n  v1: {r1[:max_len]}{'...' if len(r1) > max_len else ''}")
    print(f"\n  v2: {r2[:max_len]}{'...' if len(r2) > max_len else ''}")

langfuse.flush()

## 7. Promover v2 a producción

Una vez que v2 pasa nuestras pruebas, movemos el label `production` a v2.
Esto es equivalente a hacer un "deploy" del nuevo prompt.

In [ ]:
# Promover v2: mover el label "production" a v2
try:
    langfuse.create_prompt(
        name=prompt_name,
        prompt=PROMPT_V2,
        labels=["latest", "production"],
        type="text",
    )
    print("Prompt v2 promovido a production")
    print("\nEstado actual:")
    print("   v1 -> (sin labels activos)")
    print("   v2 -> latest, production <- ACTIVO")
except Exception as e:
    print(f"Info: {e}")

## 8. Rollback

Si v2 causa problemas, podemos hacer rollback a v1 creando una nueva versión con el contenido de v1.

> **Nota:** En Langfuse, los labels se mueven entre versiones. Cambiar el label `production` de v2 a v1 es un rollback instantáneo.

```python
# Para rollback: re-crear el prompt con el contenido de v1 y label production
langfuse.create_prompt(
    name=prompt_name,
    prompt=PROMPT_V1,  # Contenido original
    labels=["production"],  # Vuelve a ser el prompt activo
    type="text",
)
```

## 9. Concepto: promptfoo y evaluación de prompts en CI/CD

### ¿Qué es promptfoo?
Una herramienta de evaluación de prompts que permite:
- Definir test cases en YAML
- Ejecutar prompts contra un conjunto de pruebas
- Comparar versiones (A/B testing de prompts)
- Integrar en CI/CD (GitHub Actions)

### ¿Cómo encaja en el workflow?
```
Desarrollador edita prompt -> Crea nueva versión en Langfuse ->
    CI ejecuta evaluaciones -> Si pasan -> Promote a production
                             -> Si fallan -> Bloquear deploy
```

### Alternativas notebook-friendly
- **Langfuse Datasets**: test cases almacenados en Langfuse, ejecutables desde Python
- **deepeval**: librería Python para evaluación de LLMs
- **Custom**: scripts Python con assertions

> En el **Notebook 4** implementaremos evaluaciones directamente con Python y Langfuse Datasets.

## Resumen

| Concepto | Qué aprendimos |
|----------|----------------|
| **Prompt drift** | Los prompts cambian sin control -> comportamiento impredecible |
| **Versionado** | Cada cambio de prompt es una versión con historial |
| **Labels** | `production`, `latest` — control de qué versión está activa |
| **Rollback** | Cambiar label = rollback instantáneo |
| **Prompt como artefacto** | El prompt se gestiona como código: versionado, testeado, desplegado |

> **Concepto clave:** Del "edito el prompt y rezo" al "versiono, comparo, y despliego con confianza".

---

## Siguiente: [Notebook 4 - Evaluación del Agente](../day_2/04_evaluation.ipynb)